In [ ]:
import scanpy as sc
import pandas as pd
import os
import numpy as np
import re

### 预处理
This notebook is largely derived from the preprocessing followed by Lotfollahi, Mohammad, et al. "Predicting cellular responses to complex perturbations in high‐throughput screens." Molecular systems biology 19.6 (2023): e11517.

See https://github.com/facebookresearch/CPA/blob/main/preprocessing/Norman19.ipynb 

我这里直接上ncbi下载文件 参考了GEARS和CPA的预处理notebook

In [ ]:
# original data from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE133344  #filtered
adata = sc.read_10x_mtx(
    './raw/norman_filtered_raw',  
    var_names='gene_ids',
    cache=True
)

meta = pd.read_csv('./raw/norman_filtered_raw/cell_identities.csv', index_col=0)

adata.obs = adata.obs.join(meta)
adata.obs['good_coverage'] = adata.obs['good_coverage'].astype(str)

print("转换完成！")

In [ ]:
adata

In [ ]:
adata.var

In [ ]:
adata.obs

In [ ]:
# remove "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0" suggested by authors
adata = adata[adata.obs["guide_identity"] != "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0"] 
# reserve only good_coverage == True 原文没有 但感觉得这么干
adata = adata[adata.obs["good_coverage"]=="True"]

In [ ]:
needed_obs = adata.obs[["guide_identity", "UMI_count","gemgroup","number_of_cells"]].copy()
adata_new = sc.AnnData(adata.X.copy(), obs=needed_obs, var=adata.var.copy())

In [ ]:
adata_new

## prepare data

In [ ]:
# merge control
adata_new.obs["guide_merged"] = adata_new.obs["guide_identity"].astype(str)
for i in np.unique(adata_new.obs["guide_merged"]):
   m = re.match(r"NegCtrl(.*)_NegCtrl(.*)__NegCtrl(.*)_NegCtrl(.*)", i)
   if m :
        adata_new.obs["guide_merged"].replace(i,"ctrl",inplace=True)

In [ ]:
# relabel
old_pool = []
for i in np.unique(adata_new.obs["guide_merged"]):
    if i == "ctrl":
        old_pool.append(i)
        continue
    split = i.split("__")[1]
    split = split.split("_")
    for j, string in enumerate(split):
        if "NegCtrl" in split[j]:
            split[j] = "ctrl"
    if len(split) == 1:
        if split[0] in old_pool:
            print("old:",i, "new:",split[0])
        adata_new.obs["guide_merged"].replace(i,split[0],inplace=True)
        old_pool.append(split[0])
    else:
        if f"{split[0]}+{split[1]}" in old_pool:
            print("old:",i, "new:",f"{split[0]}+{split[1]}")
        adata_new.obs["guide_merged"].replace(i, f"{split[0]}+{split[1]}",inplace=True)
        old_pool.append(f"{split[0]}+{split[1]}")

In [ ]:
adata_new.obs["guide_merged"].value_counts()

In [ ]:
adata_new.write("./raw/my_norman.h5ad")

In [ ]:
adata_new = sc.read_h5ad("./raw/my_norman.h5ad")

In [ ]:
adata_new

In [ ]:
conditions = [(c.split('+')[0], c.split('+')[1]) for c in adata_new.obs['guide_merged'] if '+' in c]
conditions = [item for sublist in conditions for item in sublist]
genes_to_keep = np.unique(conditions)
genes_to_keep = genes_to_keep[genes_to_keep!="ctrl"]

In [ ]:
genes_to_keep

In [ ]:
df = pd.DataFrame(genes_to_keep, columns=['gene'])
df.to_csv('target_genes_norman.csv', index=False)